# Exploração: Sergipe, 2022

Objetivo deste notebook: conhecer a estrutura das tabelas de candidatos, receitas e resultados da Base dos Dados antes de definir filtros e regras da análise.

O recorte é pequeno de propósito (ano 2022, UF Sergipe) para as consultas serem rápidas e baratas. Nenhuma decisão metodológica é tomada aqui: o resultado serve para preencher as pendências registradas em `docs/decisoes.md`.

As queries ficam na pasta `sql/` e são executadas pelas funções de `src/conexao.py`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Permite importar o pacote src a partir da pasta notebooks/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from src.conexao import executar_query, ler_sql

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

## Teste de conexão

Confirma que o `BILLING_PROJECT_ID` do `.env` está correto e que a autenticação no Google Cloud funciona.

In [ ]:
executar_query("SELECT 1 AS teste")

## Ordem de execução

As seções seguem a ordem em que as queries devem ser rodadas, não a numeração dos arquivos:

1. `04_checagem_chaves.sql`: vem primeiro porque decide a chave de junção do projeto. Se a chave não funcionar, as demais análises que cruzam tabelas precisam ser revistas.
2. `05_resultados_valores.sql`
3. `01_candidatos_exploracao.sql`
4. `02_receitas_fontes.sql`
5. `03_receitas_repasses.sql`

## 1. Checagem das chaves de junção

Arquivo: `sql/04_checagem_chaves.sql`

Objetivo: saber se dá para juntar receitas e resultados aos candidatos pelo sequencial. Em `candidatos` a chave se chama `sequencial`; nas outras tabelas, `sequencial_candidato`. Texto vazio nas chaves é tratado como nulo.

Como ler cada linha do resultado (coluna `ordem`):

1. **Receitas com sequencial nulo:** percentual de linhas de `receitas_candidato` sem `sequencial_candidato`. Essas linhas nunca vão casar com candidatos.
2. **Resultados com sequencial nulo:** o mesmo para `resultados_candidato`.
3. **Casamento sobre o total:** percentual de linhas de receitas que encontram candidato, contando no denominador todas as linhas, inclusive as de sequencial nulo.
4. **Casamento sobre as preenchidas:** o mesmo percentual, mas só entre as linhas com `sequencial_candidato` preenchido. A diferença entre as linhas 3 e 4 mostra quanto do não casamento vem de chave nula.
5. **Sequenciais repetidos em candidatos:** quantidade de valores de `sequencial` que aparecem mais de uma vez em `candidatos` (2022, SE). Se for maior que zero, a chave sozinha não identifica uma candidatura.
6. **Não casamento por cargo:** linhas de receitas com sequencial preenchido e sem par em `candidatos`, uma linha por cargo. Se o não casamento se concentrar em cargos nacionais (ex.: presidente), a causa é o filtro `sigla_uf = 'SE'` em `candidatos`, e não um problema da chave.

Nesta exploração a junção usa só o sequencial. Na análise nacional, a chave deve ser ano + tipo_eleicao + sequencial, conforme o guia da Base dos Dados.

In [ ]:
chaves = executar_query(ler_sql("04_checagem_chaves.sql"))
chaves

## 2. Valores de resultado e turno

Arquivo: `sql/05_resultados_valores.sql`

Objetivo: ver como o resultado da eleição está codificado em cada turno, para depois definir o que conta como "eleito".

In [ ]:
resultados = executar_query(ler_sql("05_resultados_valores.sql"))
resultados

## 3. Candidaturas por cargo, partido, gênero, raça e situação

Arquivo: `sql/01_candidatos_exploracao.sql`

Objetivo: ver quais categorias existem em cada coluna (como gênero e raça estão escritos, quais situações de candidatura aparecem, quais cargos) e quantas candidaturas há em cada combinação. Isso orienta as decisões sobre quais situações e cargos entram na análise.

In [ ]:
candidatos = executar_query(ler_sql("01_candidatos_exploracao.sql"))
candidatos.head(20)

In [ ]:
# Valores distintos de cada coluna categórica
for coluna in ["cargo", "genero", "raca", "situacao"]:
    print(f"
{coluna}")
    print(candidatos.groupby(coluna, dropna=False)["n_candidaturas"].sum().sort_values(ascending=False))

## 4. Fonte, origem e natureza das receitas

Arquivo: `sql/02_receitas_fontes.sql`

Objetivo: listar as combinações de fonte, origem e natureza da receita, com quantidade de lançamentos e soma de valor. É a partir daqui que se define como identificar o dinheiro do FEFC nos dados e se as receitas estimáveis entram na conta.

In [ ]:
fontes = executar_query(ler_sql("02_receitas_fontes.sql"))
fontes

## 5. Repasses de outros candidatos e de órgãos partidários

Arquivo: `sql/03_receitas_repasses.sql`

Objetivo: medir quanto das receitas tem candidato doador identificado (`sequencial_candidato_doador` preenchido) e como se distribuem por esfera partidária do doador. Esses repasses podem fazer o mesmo recurso ser contado mais de uma vez.

In [ ]:
repasses = executar_query(ler_sql("03_receitas_repasses.sql"))
repasses

In [ ]:
# Totais por sequencial do doador preenchido ou nulo
repasses.groupby("sequencial_doador")[["n_lancamentos", "valor_total"]].sum()

## Anotações

Registre aqui o que chamou atenção e leve as decisões tomadas para `docs/decisoes.md`.